# Assignment Sesi 29 Tugas 2
Nama: Faraday Barr Fatahillah

**Tugas 2**

Anda bekerja di perusahaan Otomotif dan diberikan link [ini](https://autocatalogarchive.com/mitsubishi/) yang berisi informasi tentang beberapa mobil mitsubishi dengan berbagai macam bahasa dalam bentuk dokumen PDF. Dari katolog tersebut, gunakanlah semua file PDF yang berbahasa Indonesia atau berkode (ID), misalkan **2018 - Outlander Sport (ID)**.

Buatlah AI yang dapat yang dapat melakukan Product Search yang dapat menjawab atau mencari konteks yang cocok dengan input berikut:
- Detail spesifikasi Mitsubishi Destinator
- Mobil yang cocok untuk Travel dengan jumlah bangku atau *seating capacity* yang besar.
- Mobil untuk perjalanan jauh yang nyaman
- Mobil Mitsubishi yang irit bahan bakar
- Mobil Mitsubishi hybrid atau electric 

Lakukanlah pencarian dengan Hybrid Search dan gunakanlah Pinecone sebagai vector database.

In [31]:
import re
import os
import json
import dotenv
import pymupdf
import chromadb
import pdfplumber
import numpy as np
from tqdm import tqdm
from pathlib import Path
from rank_bm25 import BM25Okapi
from chromadb.config import Settings
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer


from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, HRFlowable
)
from reportlab.lib import colors

dotenv.load_dotenv()

True

In [ ]:
PDF_FOLDER = "./Session29_Tugas2_PDFS"

COMBINED_PDF_PATH = "mitsubishi_combined.pdf"
CHROMA_PERSIST_DIR = "./tmp/chroma_mitsubishi"
CHROMA_COLLECTION = "mitsubishi_id"

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "YOUR_PINECONE_API_KEY")
PINECONE_ENV = os.getenv("PINECONE_ENV", "us-east-1")
PINECONE_INDEX = "mitsubishi-id"

EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM = 384

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

ALPHA = 0.6
TOP_K = 5

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\bobe\AppData\Local\Temp\ipykernel_26600\3809491538.py:1: SyntaxWarning: invalid escape sequence '\S'
  PDF_FOLDER = ".\Session29_Tugas2_PDFS"


In [20]:
def extract_text_from_pdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text('text').strip()
        if text:
            pages_text.append(text)
    doc.close()

    tables_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    clean = [cell.strip() if cell else '' for cell in row]
                    if any(clean):
                        tables_text.append(' | '.join(clean))

    combined = '\n'.join(pages_text)
    if tables_text:
        combined += '\n\n--- TABLES ---\n' + '\n'.join(tables_text)

    return re.sub(r'\n{3,}', '\n\n', combined).strip()


pdf_files = sorted(Path(PDF_FOLDER).glob('*.pdf'))
print(f'Found {len(pdf_files)} PDF(s):')
for p in pdf_files:
    print(f'  • {p.name}')

Found 27 PDF(s):
  • Mitsubishi-Delica-2015-ID.pdf
  • Mitsubishi-Destinator-2025-ID.pdf
  • Mitsubishi-Eclipse-Cross-2019-ID.pdf
  • Mitsubishi-L100-EV-2024-ID.pdf
  • Mitsubishi-L300-2015-ID.pdf
  • Mitsubishi-L300-2019-ID.pdf
  • Mitsubishi-L300-2021-ID.pdf
  • Mitsubishi-Lancer-2002-ID.pdf
  • Mitsubishi-Outlander-PHEV-2019-ID.pdf
  • Mitsubishi-Outlander-Sport-2018-ID.pdf
  • Mitsubishi-Pajero-Sport-2019-ID.pdf
  • Mitsubishi-Pajero-Sport-2024-ID.pdf
  • Mitsubishi-Pajero-Sport-Elite-Edition-2024-ID.pdf
  • Mitsubishi-Triton-2019-ID.pdf
  • Mitsubishi-Triton-2022-ID.pdf
  • Mitsubishi-Triton-2024-ID.pdf
  • Mitsubishi-Triton-HDX-2025-ID.pdf
  • Mitsubishi-XFC-Concept-2023-ID-.pdf
  • Mitsubishi-XForce-2023-ID.pdf
  • Mitsubishi-Xforce-2025-ID-.pdf
  • Mitsubishi-Xpander-2020-ID.pdf
  • Mitsubishi-Xpander-2024-ID.pdf
  • Mitsubishi-Xpander-2025-ID.pdf
  • Mitsubishi-Xpander-Cross-2019-ID.pdf
  • Mitsubishi-Xpander-Cross-2022-ID.pdf
  • Mitsubishi-Xpander-Cross-2025-ID.pdf
  • Mitsu

In [21]:
brochures: list[dict] = []

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    title = pdf_path.stem
    text = extract_text_from_pdf(str(pdf_path))
    brochures.append({"title": title, "path": str(pdf_path), "text": text})
    print(f"[{title}] → {len(text):,} chars")

print(f"\nTotal brochures parsed: {len(brochures)}")

Extracting PDFs:   0%|          | 0/27 [00:00<?, ?it/s]

[Mitsubishi-Delica-2015-ID] → 0 chars


Extracting PDFs:   7%|▋         | 2/27 [00:04<00:50,  2.02s/it]

[Mitsubishi-Destinator-2025-ID] → 2,659 chars
[Mitsubishi-Eclipse-Cross-2019-ID] → 0 chars
[Mitsubishi-L100-EV-2024-ID] → 0 chars


Extracting PDFs:  19%|█▊        | 5/27 [00:05<00:22,  1.02s/it]

...

[Mitsubishi-Xforce-2025-ID-] → 11,097 chars
[Mitsubishi-Xpander-2020-ID] → 10,179 chars


Extracting PDFs:  96%|█████████▋| 26/27 [00:40<00:01,  1.47s/it]

[Mitsubishi-Xpander-2024-ID] → 1,301 chars
[Mitsubishi-Xpander-2025-ID] → 0 chars
[Mitsubishi-Xpander-Cross-2019-ID] → 0 chars
[Mitsubishi-Xpander-Cross-2022-ID] → 0 chars
[Mitsubishi-Xpander-Cross-2025-ID] → 0 chars


Extracting PDFs: 100%|██████████| 27/27 [00:41<00:00,  1.53s/it]

[Mitsubishi-Xpander-Cross-Elite-Edition-2024-ID] → 1,539 chars

Total brochures parsed: 27


In [22]:
def build_combined_pdf(brochures: list[dict], output_path: str) -> None:
    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "BrochureTitle",
        parent=styles["Heading1"],
        fontSize=18,
        textColor=colors.HexColor("#C8102E"),
        spaceAfter=12,
    )
    body_style = ParagraphStyle(
        "BrochureBody",
        parent=styles["Normal"],
        fontSize=9,
        leading=13,
        spaceAfter=4,
    )

    story = []
    for i, brochure in enumerate(brochures):
        if i > 0:
            story.append(PageBreak())

        story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor("#C8102E")))
        story.append(Spacer(1, 6))
        story.append(Paragraph(f"BROCHURE: {brochure['title']}", title_style))
        story.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
        story.append(Spacer(1, 12))

        safe_text = (
            brochure["text"]
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
        )
        for para in safe_text.split("\n\n"):
            para = para.strip()
            if para:
                story.append(Paragraph(para.replace("\n", "<br/>"), body_style))
                story.append(Spacer(1, 4))

    doc.build(story)
    print(f"Combined PDF saved → {output_path}")

build_combined_pdf(brochures, COMBINED_PDF_PATH)

Combined PDF saved → mitsubishi_combined.pdf


In [23]:
combined_text = extract_text_from_pdf(COMBINED_PDF_PATH)
print(f"Combined PDF total characters: {len(combined_text):,}")
print("\n--- Preview (first 500 chars) ---")
print(combined_text[:500])

Combined PDF total characters: 34,536

--- Preview (first 500 chars) ---
BROCHURE: Mitsubishi-Delica-2015-ID
BROCHURE: Mitsubishi-Destinator-2025-ID
DOWNLOAD HERE
NORMAL
WET
GRAVEL
TARMAC
MUD
Note: Spesifikasi dapat berubah sewaktu-waktu tanpa pemberitahuan sebelumnya.
**Tanpa engine under cover
Window film and APAR are MMKSI Authorized Standard Equipment.
Tailgate Pet Name Emblem and Floor Mat are Standard Equipment with MMC Genuine Accessories.
60.000KM
100.000KM
+ 20.000KM
3
1
EXPRESS
DESTINATOR SPECIFICATION
ULTIMATE CVT
EXCEED CVT
GLS CVT
DIMENSION & WEIGHT
Over


In [24]:
def recursive_split(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: list[str] | None = None,
) -> list[str]:
    if separators is None:
        separators = ["\n\n", "\n"]

    def _split(text: str, seps: list[str]) -> list[str]:
        if len(text) <= chunk_size:
            return [text] if text.strip() else []

        sep = seps[0] if seps else ""
        parts = text.split(sep) if sep else list(text)

        chunks: list[str] = []
        current = ""
        for part in parts:
            candidate = (current + sep + part) if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current.strip():
                    if len(current) > chunk_size and len(seps) > 1:
                        chunks.extend(_split(current, seps[1:]))
                    else:
                        chunks.append(current)
                current = part
        if current.strip():
            if len(current) > chunk_size and len(seps) > 1:
                chunks.extend(_split(current, seps[1:]))
            else:
                chunks.append(current)
        return chunks

    raw_chunks = _split(text, separators)

    if chunk_overlap == 0 or len(raw_chunks) <= 1:
        return raw_chunks

    overlapped: list[str] = [raw_chunks[0]]
    for i in range(1, len(raw_chunks)):
        tail = overlapped[-1][-chunk_overlap:]
        merged = (tail + " " + raw_chunks[i]).strip()
        overlapped.append(merged[:chunk_size + chunk_overlap])
    return overlapped

all_chunks = recursive_split(combined_text)
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c) for c in all_chunks)/len(all_chunks):.0f} chars")
print("\n--- Sample chunk ---")
print(all_chunks[0])

Total chunks: 72
Avg chunk length: 578 chars

--- Sample chunk ---
BROCHURE: Mitsubishi-Delica-2015-ID
BROCHURE: Mitsubishi-Destinator-2025-ID
DOWNLOAD HERE
NORMAL
WET
GRAVEL
TARMAC
MUD
Note: Spesifikasi dapat berubah sewaktu-waktu tanpa pemberitahuan sebelumnya.
**Tanpa engine under cover
Window film and APAR are MMKSI Authorized Standard Equipment.
Tailgate Pet Name Emblem and Floor Mat are Standard Equipment with MMC Genuine Accessories.
60.000KM
100.000KM
+ 20.000KM
3
1
EXPRESS
DESTINATOR SPECIFICATION
ULTIMATE CVT
EXCEED CVT
GLS CVT
DIMENSION & WEIGHT


In [25]:
def tag_chunks_with_source(
    chunks: list[str],
    brochures: list[dict],
) -> list[dict]:
    current_source = "unknown"
    tagged: list[dict] = []

    for i, chunk in enumerate(chunks):
        for brochure in brochures:
            if f"BROCHURE: {brochure['title']}" in chunk:
                current_source = brochure["title"]
                break
        tagged.append({
            "id":     f"chunk_{i:05d}",
            "text":   chunk.strip(),
            "source": current_source,
        })
    return tagged


tagged_chunks = tag_chunks_with_source(all_chunks, brochures)
print(f"Tagged {len(tagged_chunks)} chunks.")

from collections import Counter
dist = Counter(c["source"] for c in tagged_chunks)
for src, count in dist.most_common():
    print(f"  {src}: {count} chunks")

Tagged 72 chunks.
  Mitsubishi-Outlander-Sport-2018-ID: 23 chunks
  Mitsubishi-Xpander-2020-ID: 20 chunks
  Mitsubishi-L300-2021-ID: 7 chunks
  Mitsubishi-Delica-2015-ID: 5 chunks
  Mitsubishi-Xpander-2025-ID: 5 chunks
  Mitsubishi-Eclipse-Cross-2019-ID: 4 chunks
  Mitsubishi-L300-2019-ID: 4 chunks
  Mitsubishi-Xpander-2024-ID: 3 chunks
  Mitsubishi-Lancer-2002-ID: 1 chunks


In [26]:
print(f"Loading embedding model: {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in tagged_chunks]

print(f"Embedding {len(texts)} chunks (this may take a minute) ...")
embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f"Embeddings shape: {embeddings.shape}")

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2 ...


c:\Users\bobe\anaconda3\envs\bootcamp\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bobe\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8046.84it/s]


Embedding 72 chunks (this may take a minute) ...


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.75it/s]

Embeddings shape: (72, 384)


In [27]:
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PERSIST_DIR,
    settings=Settings(anonymized_telemetry=False),
)

try:
    chroma_client.delete_collection(CHROMA_COLLECTION)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

BATCH = 100
for start in tqdm(range(0, len(tagged_chunks), BATCH), desc="ChromaDB upsert"):
    batch = tagged_chunks[start : start + BATCH]
    collection.add(
        ids = [c["id"]   for c in batch],
        documents = [c["text"] for c in batch],
        embeddings = embeddings[start : start + BATCH].tolist(),
        metadatas = [{"source": c["source"]} for c in batch],
    )

print(f"ChromaDB collection '{CHROMA_COLLECTION}' → {collection.count()} documents")

ChromaDB upsert: 100%|██████████| 1/1 [00:00<00:00, 13.15it/s]

ChromaDB collection 'mitsubishi_id' → 72 documents


In [32]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

existing_indexes = [idx.name for idx in pc.list_indexes()]
if PINECONE_INDEX not in existing_indexes:
    pc.create_index(
        name = PINECONE_INDEX,
        dimension = EMBED_DIM,
        metric = "cosine",
        spec = ServerlessSpec(cloud="aws", region=PINECONE_ENV),
    )
    print(f"Created Pinecone index '{PINECONE_INDEX}'")
else:
    print(f"Using existing Pinecone index '{PINECONE_INDEX}'")

index = pc.Index(PINECONE_INDEX)

BATCH = 100
for start in tqdm(range(0, len(tagged_chunks), BATCH), desc="Pinecone upsert"):
    batch = tagged_chunks[start : start + BATCH]
    embeds = embeddings[start : start + BATCH]
    vectors = [
        {
            "id": chunk["id"],
            "values": emb.tolist(),
            "metadata": {"text": chunk["text"], "source": chunk["source"]},
        }
        for chunk, emb in zip(batch, embeds)
    ]
    index.upsert(vectors=vectors)

stats = index.describe_index_stats()
print(f"Pinecone index stats: {stats}")

Created Pinecone index 'mitsubishi-id'


Pinecone upsert: 100%|██████████| 1/1 [00:02<00:00,  2.86s/it]


Pinecone index stats: DescribeIndexStatsResponse(dimension=384, total_vector_count=72, metric='cosine', namespaces=1)


In [33]:
def tokenize(text: str) -> list[str]:
    return re.sub(r"[^\w\s]", " ", text.lower()).split()

corpus_tokens = [tokenize(c["text"]) for c in tagged_chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 index built over {len(corpus_tokens)} documents.")

BM25 index built over 72 documents.


In [34]:
def min_max_norm(arr: np.ndarray) -> np.ndarray:
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-9)


def hybrid_search(
    query: str,
    top_k: int = TOP_K,
    alpha: float = ALPHA,
    use_pinecone: bool = False,
) -> list[dict]:
    n = len(tagged_chunks)

    query_emb = embed_model.encode(
        [query], normalize_embeddings=True
    )[0]

    if use_pinecone:
        pass

    dense_scores_raw = embeddings @ query_emb
    dense_scores = min_max_norm(dense_scores_raw)

    query_tokens    = tokenize(query)
    bm25_scores_raw = np.array(bm25.get_scores(query_tokens))
    bm25_scores     = min_max_norm(bm25_scores_raw)

    hybrid_scores = alpha * dense_scores + (1 - alpha) * bm25_scores

    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        chunk = tagged_chunks[idx]
        results.append({
            "rank": rank,
            "id": chunk["id"],
            "source": chunk["source"],
            "hybrid_score": float(hybrid_scores[idx]),
            "dense_score": float(dense_scores_raw[idx]),
            "bm25_score": float(bm25_scores_raw[idx]),
            "text": chunk["text"],
        })
    return results


def pretty_print_results(query: str, results: list[dict]) -> None:
    sep = "=" * 80
    print(f"\n{sep}")
    print(f"QUERY: {query}")
    print(sep)
    for r in results:
        print(f"\n[Rank {r['rank']}] Source: {r['source']}")
        print(f"Hybrid={r['hybrid_score']:.4f}  Dense={r['dense_score']:.4f}  BM25={r['bm25_score']:.4f}")
        print(f"ID: {r['id']}")
        print(f"Text preview: {r['text'][:300].replace(chr(10), ' ')} ...")
    print()


print("Hybrid search function ready.")

Hybrid search function ready.


In [35]:
Q1 = "Detail spesifikasi Mitsubishi Destinator"
results_q1 = hybrid_search(Q1, top_k=TOP_K)
pretty_print_results(Q1, results_q1)


QUERY: Detail spesifikasi Mitsubishi Destinator

[Rank 1] Source: Mitsubishi-Delica-2015-ID
Hybrid=0.8531  Dense=0.5974  BM25=8.5641
ID: chunk_00000
Text preview: BROCHURE: Mitsubishi-Delica-2015-ID BROCHURE: Mitsubishi-Destinator-2025-ID DOWNLOAD HERE NORMAL WET GRAVEL TARMAC MUD Note: Spesifikasi dapat berubah sewaktu-waktu tanpa pemberitahuan sebelumnya. **Tanpa engine under cover Window film and APAR are MMKSI Authorized Standard Equipment. Tailgate Pet N ...

[Rank 2] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.6525  Dense=0.7335  BM25=1.1230
ID: chunk_00051
Text preview: IMUM TORQUE 141 Nm/4.000 rpm 1.5L MIVEC DOHC 16-VALVE Pengaturan Kursi 6 Penumpang + Sandaran Lengan Dilengkapi engine cover untuk memberikan tampilan sleek dan rapi (tipe ULTIMATE, SPORT dan EXCEED). Semua fitur berteknologi canggih milik Mitsubishi Xpander menawarkan banyak kemudahan bagi Anda dan ...

[Rank 3] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.6384  Dense=0.7233  BM25=1.0591
ID: chunk_00044
Text pre

In [36]:
Q2 = "Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity besar"
results_q2 = hybrid_search(Q2, top_k=TOP_K)
pretty_print_results(Q2, results_q2)


QUERY: Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity besar

[Rank 1] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.8437  Dense=0.4808  BM25=7.6509
ID: chunk_00061
Text preview: s *Spesifikasi sewaktu-waktu dapat berubah tanpa pemberitahuan terlebih dahulu. Fabric Seat INTERIOR Leather-wrapped Steering Wheel with Handsfree switch Urethane Steering Wheel Seating Capacity Displacement Fuel Tank Capacity Brake Assist (BA) Available GLX M/T Audio Steering Wheel Tilt & Telescopi ...

[Rank 2] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.7006  Dense=0.5107  BM25=4.1595
ID: chunk_00052
Text preview: khusus sesuai standar semua kendaraan Mitsubishi untuk memberikan keamanan terbaik dalam berkendara. 980 MPa class 590 MPa class 440 MPa class High Tensile Strength Steel Fitur yang membantu Anda mengatur laju kendaraan dan menghindari rem terkunci saat melakukan pengereman atau saat melintasi jalan ...

[Rank 3] Source: Mitsubishi-L300-2019-ID
Hybrid=0.6932  Dense=0.5994  

In [37]:
Q3 = "Mobil untuk perjalanan jauh yang nyaman"
results_q3 = hybrid_search(Q3, top_k=TOP_K)
pretty_print_results(Q3, results_q3)


QUERY: Mobil untuk perjalanan jauh yang nyaman

[Rank 1] Source: Mitsubishi-Outlander-Sport-2018-ID
Hybrid=0.9540  Dense=0.5137  BM25=7.5929
ID: chunk_00028
Text preview: atan kepada pengemudi melalui Digital Driver Display saat mundur. Mengaktifkan lampu depan kendaraan secara otomatis ketika mendeteksi kondisi minim cahaya. Membantu pengemudi mempertahankan kecepatan dan jarak aman dengan kendaraan di depan saat perjalanan jauh atau kondisi jalanan padat. Secara ot ...

[Rank 2] Source: Mitsubishi-Outlander-Sport-2018-ID
Hybrid=0.7809  Dense=0.5448  BM25=3.4966
ID: chunk_00024
Text preview: al Driver Display berpadu dengan desain dashboard hitam elegan dan sentuhan soft pad yang memberikan kesan premium. Fitur Ambient Lighting membuat suasana berkendara lebih mewah dan nyaman. 1 DRIVE MODE 1 12.3 INCH DISPLAY AUDIO AND 8 INCH DIGITAL DRIVER DISPLAY 1 COMFORT SPACE TO NAVIGATE ADVENTURE ...

[Rank 3] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.7324  Dense=0.4286  BM25=5.6050
ID: chun

In [38]:
Q4 = "Mobil Mitsubishi yang irit bahan bakar"
results_q4 = hybrid_search(Q4, top_k=TOP_K)
pretty_print_results(Q4, results_q4)


QUERY: Mobil Mitsubishi yang irit bahan bakar

[Rank 1] Source: Mitsubishi-L300-2019-ID
Hybrid=0.9721  Dense=0.6780  BM25=12.3217
ID: chunk_00010
Text preview: tangguhannya. Dengan suara mesin yang halus menghasilkan tenaga yang maksimal dan handal di tanjakan namun irit bahan bakar. Mesin Diesel 4D56 mudah perawatannya, sehingga membuat usaha Anda semakin menguntungkan. Lebih eIsien, karena memiliki ruang kargo luas yang didukung oleh mesin bertenaga sehi ...

[Rank 2] Source: Mitsubishi-Eclipse-Cross-2019-ID
Hybrid=0.8661  Dense=0.6456  BM25=9.9640
ID: chunk_00007
Text preview: . Mesin Diesel legendaris Cyclone 2.500cc sudah teruji ketangguhannya! Dengan suara mesin yang halus menghasilkan tenaga yang maksimal dan handal di tanjakan namun irit bahan bakar. Mesin Diesel 4D56 low maintenance sehingga membuat usaha Anda semakin menguntungkan. FlatDeck 4.170 1.700 1.845 4.195  ...

[Rank 3] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.7682  Dense=0.6148  BM25=7.8090
ID: chunk_00050
Text 

In [39]:
Q5 = "Mobil Mitsubishi hybrid atau electric"
results_q5 = hybrid_search(Q5, top_k=TOP_K)
pretty_print_results(Q5, results_q5)


QUERY: Mobil Mitsubishi hybrid atau electric

[Rank 1] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.7984  Dense=0.5956  BM25=3.9205
ID: chunk_00045
Text preview: p (Tipe ULTIMATE, SPORT, dan EXCEED) LED Rear Combination Lamp Front Grille and Dynamic Shield Desig Kabin Senyap dengan Peredam Suara Maksimal Attractive and Dynamic Design Setiap pembelian Mitsubishi Xpander, Anda berhak atas SMART Package yang membuat biaya perawatan reguler mobil Anda lebih hema ...

[Rank 2] Source: Mitsubishi-Xpander-2020-ID
Hybrid=0.7738  Dense=0.6431  BM25=3.0957
ID: chunk_00052
Text preview: khusus sesuai standar semua kendaraan Mitsubishi untuk memberikan keamanan terbaik dalam berkendara. 980 MPa class 590 MPa class 440 MPa class High Tensile Strength Steel Fitur yang membantu Anda mengatur laju kendaraan dan menghindari rem terkunci saat melakukan pengereman atau saat melintasi jalan ...

[Rank 3] Source: Mitsubishi-Eclipse-Cross-2019-ID
Hybrid=0.7572  Dense=0.4802  BM25=4.7311
ID: chunk_00005
Text

In [40]:
summary = {
    "Q1 - Spesifikasi Destinator":  [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q1],
    "Q2 - Seating Capacity Besar": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q2],
    "Q3 - Perjalanan Jauh Nyaman": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q3],
    "Q4 - Irit Bahan Bakar": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q4],
    "Q5 - Hybrid atau Electric": [{"rank": r["rank"], "source": r["source"], "score": round(r["hybrid_score"], 4)} for r in results_q5]
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "Q1 - Spesifikasi Destinator": [
    {
      "rank": 1,
      "source": "Mitsubishi-Delica-2015-ID",
      "score": 0.8531
    },
    {
      "rank": 2,
      "source": "Mitsubishi-Xpander-2020-ID",
      "score": 0.6525
    },
    {
      "rank": 3,
      "source": "Mitsubishi-Xpander-2020-ID",
      "score": 0.6384
    },
    {
      "rank": 4,
      "source": "Mitsubishi-Eclipse-Cross-2019-ID",
      "score": 0.6374
    },
    {
      "rank": 5,
      "source": "Mitsubishi-L300-2021-ID",
      "score": 0.5875
    }
  ],
  "Q2 - Seating Capacity Besar": [
    {
      "rank": 1,
      "source": "Mitsubishi-Xpander-2020-ID",
      "score": 0.8437
    },
    {
      "rank": 2,
      "source": "Mitsubishi-Xpander-2020-ID",
      "score": 0.7006
    },
    {
      "rank": 3,
      "source": "Mitsubishi-L300-2019-ID",
      "score": 0.6932
    },
    {
      "rank": 4,
      "source": "Mitsubishi-Eclipse-Cross-2019-ID",
      "score": 0.6855
    },
    {
      "rank": 5,
      "source"